# 🥉 Arquitetura Medalhão: Camada Bronze (Raw Data)

**Projeto:** Foodhunter (RAG System para Recomendação de Restaurantes)
**Objetivo desta camada:** A camada Bronze atua como o *Data Lake* inicial. Ela armazena os dados brutos (raw) exatamente como foram extraídos da fonte, sem nenhuma transformação, filtro ou limpeza. Ela é a nossa "fonte da verdade" histórica.

**Neste Notebook, vamos:**
1. Ingerir os dados brutos (`restaurants_with_embeddings.csv`).
2. Entender a volumetria e o schema dos dados.
3. Identificar sujeiras, valores nulos e anomalias que justifiquem as transformações da próxima etapa (Camada Silver).

In [ ]:
# Importação das bibliotecas padrão para exploração
import pandas as pd
import numpy as np
import os
import json

# Configuração para o Pandas mostrar todas as colunas sem cortar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Ingestão e Leitura dos Dados Brutos
Vamos carregar o arquivo CSV garantindo que o caminho dinâmico funcione independentemente de onde o notebook está sendo executado.

In [ ]:
# Descobre o diretório atual do notebook (pasta bronze)
diretorio_atual = os.getcwd()
diretorio_raiz = os.path.dirname(diretorio_atual)

# Monta o caminho exato para o arquivo bruto
# Como o CSV está na pasta bronze na sua árvore, lemos direto dela
caminho_bronze = os.path.join(diretorio_atual, 'restaurants_with_embeddings.csv')

print(f"Lendo dados brutos de: {caminho_bronze}")
df_bronze = pd.read_csv(caminho_bronze)

# Exibe as 3 primeiras linhas para uma checagem visual rápida
display(df_bronze.head(3))

## 2. Análise Estrutural e Tipagem (Schema)
Antes de limparmos qualquer coisa, precisamos entender o que o provedor de dados nos entregou. Quais são os tipos das colunas? Existem dados nulos?

In [ ]:
# Informações gerais do DataFrame (tipos de dados e contagem de não-nulos)
print("=== SCHEMA DO DATASET BRUTO ===")
df_bronze.info()

print("\n=== CONTAGEM DE VALORES NULOS ===")
display(df_bronze.isnull().sum())

## 3. Identificação de Dívidas Técnicas (Por que precisamos da Silver?)
A camada Bronze é "suja" por natureza. Vamos inspecionar duas colunas críticas que o sistema RAG precisa, mas que vieram em formatos problemáticos: `attributes` e `embedding`.

### 3.1. O Problema da Coluna de Atributos (JSON Stringificado)
A coluna `attributes` contém dicionários, mas o CSV os salvou como strings mal formatadas (com aspas duplas vazadas e caracteres de escape).

In [ ]:
# Pegando um exemplo aleatório da coluna de atributos
exemplo_atributo = df_bronze['attributes'].dropna().iloc[0]

print(f"TIPO DA VARIÁVEL: {type(exemplo_atributo)}")
print(f"CONTEÚDO BRUTO:\n{exemplo_atributo}")

# Tentar acessar como dicionário vai gerar erro, pois é uma string.
# A Camada Silver será responsável por fazer o 'parse' dessa string para um dict real usando ast.literal_eval.

### 3.2. O Problema do Vetor de Embeddings
O coração do nosso RAG é a coluna `embedding`. O Banco de Dados Vetorial espera uma Lista de Floats, mas o CSV transformou essa lista inteira em uma única String gigante.

In [ ]:
# Pegando um exemplo do embedding
exemplo_embedding = df_bronze['embedding'].iloc[0]

print(f"TIPO DA VARIÁVEL: {type(exemplo_embedding)}")
print(f"TAMANHO DA STRING: {len(exemplo_embedding)} caracteres")
print(f"PRIMEIROS 50 CARACTERES: {exemplo_embedding[:50]}...")

# Conclusão: A Camada Silver precisará converter essas Strings em Listas nativas do Python
# e salvar o resultado em formato .parquet, para não perdermos o tipo da variável novamente.

## 4. Conclusão da Camada Bronze
A ingestão foi bem-sucedida. O dataset contém metadados ricos sobre restaurantes e embeddings pré-calculados.

**Deveres para o pipeline da Camada Silver (`silver.py`):**
* [x] Filtrar apenas restaurantes em operação (`is_open == 1`).
* [x] Converter a coluna `attributes` de String para Dicionário Python.
* [x] Extrair features estruturadas do dicionário (ex: `has_delivery`, `price_range`, `has_outdoor`).
* [x] Converter a coluna `embedding` de String para Listas nativas.
* [x] Descartar colunas desnecessárias.
* [x] Salvar o resultado final em formato columnar (`.parquet`) para preservar os *data types*.